# 合同台账与票款执行一致性校验模型 · 逐步演示

**用法**：在 `演示/` 目录下打开本笔记本，或在项目根目录打开后直接运行第 1 个单元（会自动切到根目录）。

按顺序执行每个单元，每一步都有可验收的输出。共 4 步，约 6 分钟。

> 依赖：仅 Python 3.9+ 标准库；生成 Excel 总表需要 openpyxl（缺失会自动跳过，不影响其余步骤）。

In [ ]:
import os, sys, subprocess, csv

# 允许从 演示/ 或项目根目录打开
if os.path.basename(os.getcwd()) == "演示":
    os.chdir("..")
print("工作目录:", os.getcwd())
print("Python:", sys.version.split()[0])
need = ["code/gen_mock_data.py", "code/run_model.py", "code/run_sql_check.py",
        "code/build_report.py", "data/raw/ods_siku_contract.csv"]
missing = [p for p in need if not os.path.exists(p)]
print("缺失文件:", missing or "无")

## STEP 1 · 生成 mock 数据

生成两张表：司库 16 合同表（87 行）、非司库 11 发票及支付信息表（90 行），以及参数文件。

In [ ]:
!python3 code/gen_mock_data.py

In [ ]:
def peek(path, n=5, cols=None):
    with open(path, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    cols = cols or list(rows[0].keys())[:6]
    print(" | ".join(cols))
    print("-" * 72)
    for r in rows[:n]:
        print(" | ".join(str(r[c])[:18] for c in cols))
    print(f"共 {len(rows)} 行\n")

peek("data/raw/ods_siku_contract.csv",
     cols=["CONTRACT_CODE", "CONTRACT_AMOUNT", "SETTLEMENT_AMOUNT", "FRAMEWORK_CONTRACT"])
peek("data/raw/ods_nonsiku_invoice_pay.csv",
     cols=["合同编号", "累计发票金额", "累计支付金额", "状态"])

## STEP 2 · 运行规则引擎

清洗 → 特征 → R1 / R2 判定 → 输出 V2.1 四类对象。末尾 6 项自检必须全部 PASS。

In [ ]:
!python3 code/run_model.py

In [ ]:
with open("out/candidate.csv", encoding="utf-8-sig") as f:
    cand = list(csv.DictReader(f))
print(f"Candidate {len(cand)} 条\n")
print(f"{'candidate_id':<16}{'risk_type':<28}{'level':<8}{'score':<7}summary")
print("-" * 96)
for r in cand[:6]:
    print(f"{r['candidate_id']:<16}{r['risk_type']:<28}{r['risk_level']:<8}"
          f"{r['risk_score']:<7}{r['risk_summary'][:44]}")

## STEP 3 · SQL 与 Python 双实现校验

SQLite 载入 CSV → 执行 `sql/sqlite/*.sql` → 特征表逐行比对、线索集合差集比对。
结论必须是「通过」。

In [ ]:
!python3 code/run_sql_check.py

## STEP 4 · 生成可交互报告

产出单文件离线页 `report/任务报告.html`，双击即开。

In [ ]:
!python3 code/build_report.py

In [ ]:
import sys, os
p = os.path.abspath("report/任务报告.html")
print("报告路径:", p, "|", round(os.path.getsize(p)/1024), "KB")
if sys.platform == "darwin":
    os.system(f'open "{p}"')
elif sys.platform.startswith("win"):
    os.system(f'start "" "{p}"')
else:
    print("请手动用浏览器打开上面的路径")

## 收尾 · 三个口径提醒

1. **R1 看不到「应付」**：应付已确认但票、款都未发生时差额为 0，本模型不报，那属于应付挂账。
2. **红冲口径待确认**：11 表累计发票金额是否已剔除红冲，规范未明确；若含红冲，B2 虚高会产生「多开票」误报。
3. **阈值是演示值**：1 万元 + 5% 来自内控经验，须用 POC 核查结论回标。

现场若要演示阈值敏感性，打开报告拖「相对门槛」滑杆：拉到 0，`CG-202606`（差 300 万但仅占 3.61%）会出现在新增列表。